In [1]:
!pip install pandas pymongo fastparquet

   ---------------------------------------- 0.0/673.3 kB ? eta -:--:--
   --------------- ------------------------ 262.1/673.3 kB ? eta -:--:--
   ---------------------------------------- 673.3/673.3 kB 3.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 3.5 MB/s eta 0:00:01
   ------------------ --------------------- 0.8/1.7 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 2.7 MB/s  0:00:00

   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   -------------------------- ------------- 2/3 [fastparquet]
   -------------------------- ------------- 2/3 [fastparquet]
   ---------------------------------------- 3/3 [fastparquet]




[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pymongo import MongoClient

# --- Configuration ---
PARQUET_INPUT_PATH = r"C:/latMongo/customer_data.parquet"
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB = "AdventureSales"
MONGO_COLLECTION = "customerdist_1910a"

# 1. Read the data from the Parquet file into a Pandas DataFrame
try:
    df = pd.read_parquet(PARQUET_INPUT_PATH)
    print(f"Successfully loaded {len(df)} rows from Parquet file.")
except Exception as e:
    print(f"Error reading Parquet file: {e}")
    exit()

# 2. Clean the data to satisfy MongoDB's unique _id requirement
# Remove rows where kode_iap is empty or null, as seen in your previous output
df_cleaned = df.dropna(subset=['kode_iap'])
df_cleaned = df_cleaned[df_cleaned['kode_iap'] != '']
df_cleaned = df_cleaned[df_cleaned['kode_iap'] != '//']

print(f"Number of valid rows after cleaning: {len(df_cleaned)}")

# 3. Prepare data for MongoDB insertion
# MongoDB uses '_id' as the primary key. We map the unique 'kode_iap' column to '_id'.

# Convert DataFrame to a list of dictionaries (MongoDB documents)
data_to_insert = df_cleaned.to_dict(orient='records')

# Rename the key in each document from 'kode_iap' to '_id'
for doc in data_to_insert:
    doc['_id'] = doc.pop('kode_iap')


# 4. Connect to MongoDB and insert the data
try:
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB]
    collection = db[MONGO_COLLECTION]
    
    # Optional: Clear the collection before inserting (like Spark's "overwrite" mode)
    collection.delete_many({}) 
    
    # Perform the bulk insert operation
    collection.insert_many(data_to_insert)
    
    print(f"Successfully inserted {len(data_to_insert)} documents into MongoDB.")
    
except Exception as e:
    print(f"An error occurred during MongoDB insertion: {e}")
finally:
    if client:
        client.close()



Successfully loaded 1237180 rows from Parquet file.
Number of valid rows after cleaning: 1237179
Successfully inserted 1237179 documents into MongoDB.


In [7]:
import pymongo
import pandas as pd
from pymongo import MongoClient

# --- Configuration ---
PARQUET_INPUT_PATH = r"C:/latMongo/customer_data.parquet"
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB = "AdventureSales"
MONGO_COLLECTION = "customerdist_1910a"

# 1. Read the data from the Parquet file into a Pandas DataFrame
try:
    df = pd.read_parquet(PARQUET_INPUT_PATH)
    print(f"Successfully loaded {len(df)} rows from Parquet file.")
except Exception as e:
    print(f"Error reading Parquet file: {e}")
    exit()

# 2. Clean the data to satisfy MongoDB's unique _id requirement
# Remove rows where kode_iap is empty or null/invalid, as MongoDB _id cannot be empty
df_cleaned = df.dropna(subset=['kode_iap'])
df_cleaned = df_cleaned[df_cleaned['kode_iap'] != '']
df_cleaned = df_cleaned[df_cleaned['kode_iap'] != '//']

print(f"Number of valid rows after cleaning: {len(df_cleaned)}")

# 3. Prepare data for MongoDB insertion
# Convert DataFrame to a list of dictionaries (MongoDB documents)
data_to_insert = df_cleaned.to_dict(orient='records')

# Add the '_id' field to each document by COPYING the 'kode_iap' value
for doc in data_to_insert:
    doc['_id'] = doc['kode_iap'] # This line was changed from doc.pop('kode_iap')


# 4. Connect to MongoDB and insert the data
try:
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB]
    collection = db[MONGO_COLLECTION]
    
    # Clear the collection before inserting
    collection.delete_many({}) 
    
    # Perform the bulk insert operation
    collection.insert_many(data_to_insert)
    
    print(f"Successfully inserted {len(data_to_insert)} documents into MongoDB.")
        # --- ADD THIS CODE TO CREATE THE INDEX ---
    print("Creating unique index on the 'kode_iap' field...")
    collection.create_index(
        [('kode_iap', pymongo.ASCENDING)], 
        unique=True
    )
    print("Index created successfully.")
    
except Exception as e:
    print(f"An error occurred during MongoDB insertion: {e}")
finally:
    if client:
        client.close()



Successfully loaded 1237180 rows from Parquet file.
Number of valid rows after cleaning: 1237179
Successfully inserted 1237179 documents into MongoDB.
Creating unique index on the 'kode_iap' field...
Index created successfully.


In [ ]:
print(spark.version)